In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px

In [8]:
#lectura de los datasets
owid = pd.read_csv('../data/owid-energy-data.csv')
kaggle = pd.read_csv('../data/global-data-on-sustainable-energy.csv')

In [9]:
owid.head()

,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


In [10]:
kaggle.head()

,Entity,Year,Access to electricity (% of population),Access to clean fuels for cooking,Renewable-electricity-generating-capacity-per-capita,Financial flows to developing countries (US $),Renewable energy share in the total final energy consumption (%),Electricity from fossil fuels (TWh),Electricity from nuclear (TWh),Electricity from renewables (TWh),...,Primary energy consumption per capita (kWh/person),Energy intensity level of primary energy (MJ/$2017 PPP GDP),Value_co2_emissions_kt_by_country,Renewables (% equivalent primary energy),gdp_growth,gdp_per_capita,Density\n(P/Km2),Land Area(Km2),Latitude,Longitude
0,Afghanistan,2000,1.613591,6.2,9.22,20000.0,44.99,0.16,0.0,0.31,...,302.59482,1.64,760.000000,NaN,NaN,NaN,60,652230.0,33.93911,67.709953
1,Afghanistan,2001,4.074574,7.2,8.86,130000.0,45.60,0.09,0.0,0.50,...,236.89185,1.74,730.000000,NaN,NaN,NaN,60,652230.0,33.93911,67.709953
2,Afghanistan,2002,9.409158,8.2,8.47,3950000.0,37.83,0.13,0.0,0.56,...,210.86215,1.40,1029.999971,NaN,NaN,179.426579,60,652230.0,33.93911,67.709953
3,Afghanistan,2003,14.738506,9.5,8.09,25970000.0,36.66,0.31,0.0,0.63,...,229.96822,1.40,1220.000029,NaN,8.832278,190.683814,60,652230.0,33.93911,67.709953
4,Afghanistan,2004,20.064968,10.9,7.75,NaN,44.24,0.33,0.0,0.56,...,204.23125,1.20,1029.999971,NaN,1.414118,211.382074,60,652230.0,33.93911,67.709953


# ─── 2. PREPARACION CLAVE ANTES DEL MERGE ────────────────────────────

In [14]:
kaggle = kaggle.rename(columns={
    "Entity": "country",
    "Year":   "year",
    "Access to electricity (% of population)":                    "access_to_electricity",
    "Access to clean fuels for cooking":                          "access_clean_fuels_cooking",
    "Renewable-electricity-generating-capacity-per-capita":       "renew_elec_capacity_per_capita",
    "Financial flows to developing countries (US $)":            "financial_flows_usd",
    "Renewable energy share in the total final energy consumption (%)": "renewable_share_of_total_energy",
    "Electricity from fossil fuels (TWh)":                        "elec_from_fossil_twh",
    "Electricity from nuclear (TWh)":                             "elec_from_nuclear_twh",
    "Electricity from renewables (TWh)":                          "elec_from_renewables_twh",
    "Low-carbon electricity (% electricity)":                     "low_carbon_elec_pct",
    "Primary energy consumption per capita (kWh/person)":         "primary_energy_per_capita_kwh",
    "Energy intensity level of primary energy (MJ/$2017 PPP GDP)": "energy_intensity_primary_energy",
    "Value_co2_emissions_kt_by_country":                          "co2_emissions_kt",
    "Renewables (% equivalent primary energy)":                   "renewables_pct_primary_energy",
    "gdp_growth":       "gdp_growth",
    "gdp_per_capita":   "gdp_per_capita",
    "Density\\n(P/Km2)": "population_density",
    "Land Area(Km2)":   "land_area_km2",
    "Latitude":         "latitude",
    "Longitude":        "longitude",
})


In [15]:
# ── Filtrar OWID al rango 2000–2020 (mismo que Kaggle) ────────────────────
owid_filtered = owid[(owid["year"] >= 2000) & (owid["year"] <= 2020)].copy()

In [16]:
# ── Merge inner por country + year ────────────────────────────────────────
merged = pd.merge(owid_filtered, kaggle, on=["country", "year"], how="inner")

In [17]:
# Mapa simplificado de región por país
REGION_MAP = {
    # América Latina y el Caribe
    "Argentina":"Latin America & Caribbean","Bolivia":"Latin America & Caribbean",
    "Brazil":"Latin America & Caribbean","Chile":"Latin America & Caribbean",
    "Colombia":"Latin America & Caribbean","Costa Rica":"Latin America & Caribbean",
    "Cuba":"Latin America & Caribbean","Dominican Republic":"Latin America & Caribbean",
    "Ecuador":"Latin America & Caribbean","El Salvador":"Latin America & Caribbean",
    "Guatemala":"Latin America & Caribbean","Haiti":"Latin America & Caribbean",
    "Honduras":"Latin America & Caribbean","Jamaica":"Latin America & Caribbean",
    "Mexico":"Latin America & Caribbean","Nicaragua":"Latin America & Caribbean",
    "Panama":"Latin America & Caribbean","Paraguay":"Latin America & Caribbean",
    "Peru":"Latin America & Caribbean","Trinidad and Tobago":"Latin America & Caribbean",
    "Uruguay":"Latin America & Caribbean","Venezuela":"Latin America & Caribbean",
    # América del Norte
    "Canada":"North America","United States":"North America",
    # Europa
    "Albania":"Europe","Austria":"Europe","Belgium":"Europe","Bulgaria":"Europe",
    "Croatia":"Europe","Cyprus":"Europe","Czech Republic":"Europe","Denmark":"Europe",
    "Estonia":"Europe","Finland":"Europe","France":"Europe","Germany":"Europe",
    "Greece":"Europe","Hungary":"Europe","Iceland":"Europe","Ireland":"Europe",
    "Italy":"Europe","Latvia":"Europe","Lithuania":"Europe","Luxembourg":"Europe",
    "Malta":"Europe","Netherlands":"Europe","Norway":"Europe","Poland":"Europe",
    "Portugal":"Europe","Romania":"Europe","Serbia":"Europe","Slovakia":"Europe",
    "Slovenia":"Europe","Spain":"Europe","Sweden":"Europe","Switzerland":"Europe",
    "Ukraine":"Europe","United Kingdom":"Europe",
    # Asia del Este y Pacífico
    "Australia":"East Asia & Pacific","China":"East Asia & Pacific",
    "Indonesia":"East Asia & Pacific","Japan":"East Asia & Pacific",
    "Malaysia":"East Asia & Pacific","Myanmar":"East Asia & Pacific",
    "New Zealand":"East Asia & Pacific","Philippines":"East Asia & Pacific",
    "Singapore":"East Asia & Pacific","South Korea":"East Asia & Pacific",
    "Thailand":"East Asia & Pacific","Vietnam":"East Asia & Pacific",
    # Asia del Sur
    "Bangladesh":"South Asia","India":"South Asia","Nepal":"South Asia",
    "Pakistan":"South Asia","Sri Lanka":"South Asia",
    # Oriente Medio y Norte de África
    "Algeria":"Middle East & North Africa","Egypt":"Middle East & North Africa",
    "Iran":"Middle East & North Africa","Iraq":"Middle East & North Africa",
    "Jordan":"Middle East & North Africa","Kuwait":"Middle East & North Africa",
    "Lebanon":"Middle East & North Africa","Libya":"Middle East & North Africa",
    "Morocco":"Middle East & North Africa","Oman":"Middle East & North Africa",
    "Qatar":"Middle East & North Africa","Saudi Arabia":"Middle East & North Africa",
    "Tunisia":"Middle East & North Africa","United Arab Emirates":"Middle East & North Africa",
    "Yemen":"Middle East & North Africa",
    # África Subsahariana
    "Angola":"Sub-Saharan Africa","Cameroon":"Sub-Saharan Africa",
    "Congo":"Sub-Saharan Africa","Democratic Republic of Congo":"Sub-Saharan Africa",
    "Ethiopia":"Sub-Saharan Africa","Ghana":"Sub-Saharan Africa",
    "Kenya":"Sub-Saharan Africa","Mozambique":"Sub-Saharan Africa",
    "Nigeria":"Sub-Saharan Africa","Senegal":"Sub-Saharan Africa",
    "South Africa":"Sub-Saharan Africa","Sudan":"Sub-Saharan Africa",
    "Tanzania":"Sub-Saharan Africa","Uganda":"Sub-Saharan Africa",
    "Zambia":"Sub-Saharan Africa","Zimbabwe":"Sub-Saharan Africa",
    # Europa del Este y Asia Central
    "Azerbaijan":"Eastern Europe & Central Asia","Belarus":"Eastern Europe & Central Asia",
    "Georgia":"Eastern Europe & Central Asia","Kazakhstan":"Eastern Europe & Central Asia",
    "Kyrgyzstan":"Eastern Europe & Central Asia","Moldova":"Eastern Europe & Central Asia",
    "Russia":"Eastern Europe & Central Asia","Tajikistan":"Eastern Europe & Central Asia",
    "Turkmenistan":"Eastern Europe & Central Asia","Uzbekistan":"Eastern Europe & Central Asia",
}

In [18]:
merged["region"] = merged["country"].map(REGION_MAP).fillna("Other")

In [20]:
# ── Guardar ───────────────────────────────────────────────────────────────
out_path = "../data/merged.csv"
merged.to_csv(out_path, index=False)

print(f"\n✅ Merge completado:")
print(f"   Filas:    {merged.shape[0]:,}")
print(f"   Columnas: {merged.shape[1]}")
print(f"   Países:   {merged['country'].nunique()}")
print(f"   Años:     {merged['year'].min()} – {merged['year'].max()}")
print(f"   Guardado: {out_path}")


✅ Merge completado:
   Filas:    3,649
   Columnas: 150
   Países:   176
   Años:     2000 – 2020
   Guardado: ../data/merged.csv


# ─── 3. LIMPIEZA POST MERGE ────────────────────────────

In [36]:
df = pd.read_csv('../data/merged.csv')
print('Shape:', df.shape)
df.head()

Shape: (3649, 150)


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,energy_intensity_primary_energy,co2_emissions_kt,renewables_pct_primary_energy,gdp_growth,gdp_per_capita,population_density,land_area_km2,latitude,longitude,region
0,Afghanistan,2000,AFG,20130334.0,1.128379e+10,NaN,NaN,NaN,NaN,0.0,...,1.64,760.000000,NaN,NaN,NaN,60,652230.0,33.93911,67.709953,Other
1,Afghanistan,2001,AFG,20284303.0,1.102127e+10,NaN,NaN,NaN,NaN,0.0,...,1.74,730.000000,NaN,NaN,NaN,60,652230.0,33.93911,67.709953,Other
2,Afghanistan,2002,AFG,21378123.0,1.880487e+10,NaN,NaN,NaN,NaN,0.0,...,1.40,1029.999971,NaN,NaN,179.426579,60,652230.0,33.93911,67.709953,Other
3,Afghanistan,2003,AFG,22733053.0,2.107434e+10,NaN,NaN,NaN,NaN,0.0,...,1.40,1220.000029,NaN,8.832278,190.683814,60,652230.0,33.93911,67.709953,Other
4,Afghanistan,2004,AFG,23560656.0,2.233257e+10,NaN,NaN,NaN,NaN,0.0,...,1.20,1029.999971,NaN,1.414118,211.382074,60,652230.0,33.93911,67.709953,Other


# ═══════════════════════════════════════════════════════════════════════════
# COLUMNAS CLAVE QUE NECESITA EL DASHBOARD
# ═══════════════════════════════════════════════════════════════════════════

In [37]:
KEY_COLS = [
    'renewable_share_of_total_energy',
    'carbon_intensity_elec',
    'gdp_per_capita',
    'access_to_electricity',
    'fossil_share_elec',
    'energy_per_capita',
    'coal_electricity',
    'gas_electricity',
    'nuclear_electricity',
    'solar_electricity',
    'wind_electricity',
    'hydro_electricity',
    'energy_intensity_primary_energy',
]

print('\n--- NULOS ANTES ---')
for col in KEY_COLS:
    n = df[col].isna().sum()
    print(f'  {n:3d} ({n/len(df)*100:.1f}%)  {col}')


--- NULOS ANTES ---
  194 (5.3%)  renewable_share_of_total_energy
   21 (0.6%)  carbon_intensity_elec
  282 (7.7%)  gdp_per_capita
   10 (0.3%)  access_to_electricity
   21 (0.6%)  fossil_share_elec
    0 (0.0%)  energy_per_capita
   54 (1.5%)  coal_electricity
   94 (2.6%)  gas_electricity
   78 (2.1%)  nuclear_electricity
   24 (0.7%)  solar_electricity
  177 (4.9%)  wind_electricity
   84 (2.3%)  hydro_electricity
  207 (5.7%)  energy_intensity_primary_energy


In [38]:
# Imputación completa
df_clean = df.sort_values(['country', 'year']).reset_index(drop=True)

In [39]:
# Interpolación lineal por país + ffill + bfill
# Resuelve gaps parciales y el problema del año 2020 que falta en Kaggle
INTERP_COLS = [
    'renewable_share_of_total_energy',
    'carbon_intensity_elec',
    'gdp_per_capita',
    'access_to_electricity',
    'fossil_share_elec',
    'energy_intensity_primary_energy',
]
for col in INTERP_COLS:
    df_clean[col] = (
        df_clean.groupby('country')[col]
        .transform(lambda x: x.interpolate(method='linear').ffill().bfill())
    )

In [40]:
# Mix eléctrico — fill 0 si el país nunca generó, interpolación si tiene gaps
MIX_COLS = [
    'coal_electricity', 'gas_electricity', 'nuclear_electricity',
    'solar_electricity', 'wind_electricity', 'hydro_electricity',
]
for col in MIX_COLS:
    nunca = df_clean.groupby('country')[col].transform(lambda x: x.isna().all())
    df_clean.loc[nunca, col] = 0
    df_clean[col] = (
        df_clean.groupby('country')[col]
        .transform(lambda x: x.interpolate(method='linear').ffill().bfill())
    )

In [42]:
# Nnulos residuales (países sin NINGÚN dato histórico) , rellenar con la mediana de su región por año
REGION_MEDIAN_COLS = [
    'renewable_share_of_total_energy',
    'carbon_intensity_elec',
    'gdp_per_capita',
    'access_to_electricity',
    'fossil_share_elec',
    'energy_intensity_primary_energy',
]
for col in REGION_MEDIAN_COLS:
    mediana_region = df_clean.groupby(['region', 'year'])[col].transform('median')
    df_clean[col] = df_clean[col].fillna(mediana_region)

print('--- NULOS DESPUÉS ---')
for col in KEY_COLS:
    n = df_clean[col].isna().sum()
    flag = '✅' if n == 0 else '⚠️ '
    print(f'  {flag} {n:3d} ({n/len(df_clean)*100:.1f}%)  {col}')

df_clean.to_csv('../data/merged_imputed.csv', index=False)
print('\n Guardado: merged_imputed.csv')

--- NULOS DESPUÉS ---
  ✅   0 (0.0%)  renewable_share_of_total_energy
  ✅   0 (0.0%)  carbon_intensity_elec
  ✅   0 (0.0%)  gdp_per_capita
  ✅   0 (0.0%)  access_to_electricity
  ✅   0 (0.0%)  fossil_share_elec
  ✅   0 (0.0%)  energy_per_capita
  ✅   0 (0.0%)  coal_electricity
  ✅   0 (0.0%)  gas_electricity
  ✅   0 (0.0%)  nuclear_electricity
  ✅   0 (0.0%)  solar_electricity
  ✅   0 (0.0%)  wind_electricity
  ✅   0 (0.0%)  hydro_electricity
  ✅   0 (0.0%)  energy_intensity_primary_energy

 Guardado: merged_imputed.csv


## Cargar dataset final

In [2]:
df = pd.read_csv('../data/merged_imputed.csv')
df.head()

,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,energy_intensity_primary_energy,co2_emissions_kt,renewables_pct_primary_energy,gdp_growth,gdp_per_capita,population_density,land_area_km2,latitude,longitude,region
0,Afghanistan,2000,AFG,20130334.0,1.128379e+10,NaN,NaN,NaN,NaN,0.0,...,1.64,760.000000,NaN,NaN,179.426579,60,652230.0,33.93911,67.709953,Other
1,Afghanistan,2001,AFG,20284303.0,1.102127e+10,NaN,NaN,NaN,NaN,0.0,...,1.74,730.000000,NaN,NaN,179.426579,60,652230.0,33.93911,67.709953,Other
2,Afghanistan,2002,AFG,21378123.0,1.880487e+10,NaN,NaN,NaN,NaN,0.0,...,1.40,1029.999971,NaN,NaN,179.426579,60,652230.0,33.93911,67.709953,Other
3,Afghanistan,2003,AFG,22733053.0,2.107434e+10,NaN,NaN,NaN,NaN,0.0,...,1.40,1220.000029,NaN,8.832278,190.683814,60,652230.0,33.93911,67.709953,Other
4,Afghanistan,2004,AFG,23560656.0,2.233257e+10,NaN,NaN,NaN,NaN,0.0,...,1.20,1029.999971,NaN,1.414118,211.382074,60,652230.0,33.93911,67.709953,Other


### **Pregunta 5** - _Ranking de consumo_
¿Cómo cambió el ranking de los doce mayores consumidores de energía per cápita entre 2000 y 2020? ¿Qué países subieron y cuáles bajaron posiciones?

In [5]:
top12 = (
    df[df["year"].between(2000, 2020)]
    .groupby("year")[["country", "year", "energy_per_capita"]]
    .apply(lambda g: g.nlargest(12, "energy_per_capita"))
    .reset_index(drop=True)
)
top12["rank"] = top12.groupby("year")["energy_per_capita"].rank(
    ascending=False, method="first"
)

fig5 = px.line(
    top12, x="year", y="rank", color="country", markers=True,
    title="Qatar lideró el ranking de consumo energético per cápita entre 2000 y 2020"
)
fig5.update_yaxes(autorange="reversed", dtick=1, title="Ranking")
fig5.update_xaxes(dtick=2, title="Año")
fig5.show()

### **Pregunta 6** -  _Mix eléctrico por país_
Para el país que el evaluador seleccione, ¿cuál fue su mix de generación eléctrica por fuente (carbón, gas, nuclear, solar, eólica, hidro) en el año de mayor producción renovable?

In [15]:
pais = "Peru"
fuentes = ["coal_electricity", "gas_electricity", "nuclear_electricity", "solar_electricity", "wind_electricity", "hydro_electricity"]
etiquetas = ["Carbón", "Gas", "Nuclear", "Solar", "Eólica", "Hidro"]

datos_pais = df[df["country"] == pais].copy()
anio_max_ren = datos_pais.loc[
    datos_pais["renewables_electricity"].idxmax(), "year"
]
fila = datos_pais[datos_pais["year"] == anio_max_ren].iloc[0]
valores = [fila[f] for f in fuentes]
pct = [v / sum(valores) * 100 if sum(valores) > 0 else 0 for v in valores]

tabla_mix = pd.DataFrame({"fuente": etiquetas, "porcentaje": pct, "categoria": ""})
tabla_mix = tabla_mix.sort_values("porcentaje", ascending=False)

fig6 = px.bar(
    tabla_mix, x="porcentaje", y="categoria", color="fuente", orientation="h",
    barmode="stack", text_auto=".1f",
    title=f"Mix eléctrico ({pais} - {int(anio_max_ren)})",
    labels={"porcentaje": "Porcentaje (%)", "categoria": ""},
    category_orders={"fuente": tabla_mix["fuente"].tolist()}
)
fig6.update_xaxes(range=[0, 100])
fig6.show()